In [ ]:
#| default_exp ext

# Extension transport
> Drive the user's everyday Chrome through a companion extension

`CDP.launch` and `CDP.connect` need either a fresh browser profile or a debugging flag the user must switch on. A companion extension avoids both: it holds the `chrome.debugger` end inside the user's normal Chrome, and forwards CDP as JSON frames over a websocket. Since extensions cannot accept inbound connections, the extension always *dials out* to a peer. This module is that peer: `ExtChannel` listens on a local port for the extension to dial in, and `ExtCDP` speaks CDP over the connection. The same frame protocol also runs through a relay server when the kernel can't listen anywhere the browser can reach (that's what `solvecdp` does, meeting the extension at a solveit server's `/wsx`).

In [ ]:
#| export
from fastcore.utils import *
import asyncio, json, websockets
from urllib.parse import urlparse, parse_qsl
from fastcdp.core import *
import fastcdp.core

In [ ]:
from fastcore.test import test_eq
import httpx

In [ ]:
#| export
def _probe_ok(conn, req):
    "Answer the extension's plain-HTTP port probe instead of attempting a handshake"
    if 'Upgrade' not in req.headers: return conn.respond(200, 'fastcdp\n')

class ExtChannel:
    "Listen for the extension to dial in, then speak duplex JSON frames with it"
    def __init__(self): self._replies,self.events,self._id = {},asyncio.Queue(),0

    @classmethod
    async def listen(cls,
        port:int=34654, # Port to listen on (0 picks a free one; see `port`)
        host:str='127.0.0.1', # Interface to bind
        token:str=None # If set, the extension must dial with a matching `?token=`
    ):
        "Start listening for the extension to dial `ws://host:port/?token=`"
        self = cls()
        self._token = token
        self._peer = asyncio.get_running_loop().create_future()
        self._server = await websockets.serve(self._handler, host, port, process_request=_probe_ok)
        return self

    async def _handler(self, ws):
        q = dict(parse_qsl(urlparse(ws.request.path).query))
        if self._token and q.get('token') != self._token: return await ws.close(4003, 'bad token')
        self._ws = ws
        if not self._peer.done(): self._peer.set_result(ws)
        try:
            async for frame in ws:
                msg = dict2obj(json.loads(frame))
                if (f := self._replies.pop(msg.get('id'), None)): f.set_result(msg)
                else: await self.events.put(msg)
        except websockets.ConnectionClosed: pass

    async def wait_peer(self, timeout:float=None):
        "Wait until an extension has connected"
        await asyncio.wait_for(asyncio.shield(self._peer), timeout)

    async def send(self, **msg): await self._ws.send(json.dumps(msg))

    async def request(self, timeout:int=15, **msg):
        "Send `msg` with a correlation `id` and await the reply frame bearing that id"
        self._id += 1
        mid = msg['id'] = self._id
        f = asyncio.get_running_loop().create_future()
        self._replies[mid] = f
        try:
            await self.send(**msg)
            return await asyncio.wait_for(f, timeout)
        finally: self._replies.pop(mid, None)

    @property
    def port(self): return self._server.sockets[0].getsockname()[1]

    async def close(self):
        self._server.close()
        await self._server.wait_closed()

    @property
    def is_open(self): return getattr(self, '_ws', None) is not None and self._ws.state.name == 'OPEN'

Frames are JSON dicts, in four kinds. Commands carry a correlation `id` plus `method`/`params` (and `tabId` for tab-scoped ones); the extension answers each with `{id, result}` or `{id, error}`. Lifecycle *actions* (`{id, action}`: `new-tab`, `attach`, `get-targets`, `detach`, `close-tab`) cover operations `chrome.debugger` doesn't expose as CDP. Debugger events arrive as `{method, params, tabId}` with no `id`, and keepalive pings have neither `id` nor `method` and are ignored by consumers. `request` handles the correlation; anything that isn't a reply lands in `events`.

`token` is the trust check: the listener hands full browser control to whoever dials in, so outside of throwaway settings the extension should be configured with a shared secret, and only matching dials are accepted.

To exercise everything without a browser, a scripted stand-in plays the extension: it dials the listener and answers frames the way the real one would, faking a single tab 99. Method frames are echoed back so tests can see exactly what arrived, and enabling the `Page` domain emits one debugger event, as the real extension does when a page loads.

In [ ]:
async def fake_ext(url):
    "A stand-in extension: one fake tab 99, canned replies, an event on `Page.enable`"
    try:
        async with websockets.connect(url) as ws:
            async for frame in ws:
                m = json.loads(frame)
                if m.get('action')=='new-tab': r = dict(tabId=99)
                elif m.get('action')=='get-targets': r = [dict(type='page', tabId=99, url=m.get('url'))]
                elif m.get('action')=='close-tab': r = {}
                else: r = dict(method=m['method'], tabId=m.get('tabId'))
                await ws.send(json.dumps(dict(id=m['id'], result=r)))
                if m.get('method')=='Page.enable':
                    await ws.send(json.dumps(dict(method='Page.loadEventFired', params={}, tabId=m['tabId'])))
    except websockets.ConnectionClosed: pass

In [ ]:
chan = await ExtChannel.listen(port=0, token='s3cret')
fake = asyncio.create_task(fake_ext(f'ws://localhost:{chan.port}/?token=s3cret'))
await chan.wait_peer(5)
assert chan.is_open

A dial with the wrong token is refused before any frame flows:

In [ ]:
bad = await websockets.connect(f'ws://localhost:{chan.port}/?token=wrong')
try: await asyncio.wait_for(bad.recv(), 5); raise AssertionError('expected close')
except websockets.ConnectionClosed as e: test_eq(e.rcvd.code, 4003)

The listener also answers plain HTTP: the extension probes the port with a cheap `fetch` before dialing the websocket, so a port with nothing listening never spams its console with failed connections.

In [ ]:
async with httpx.AsyncClient() as c: r = await c.get(f'http://localhost:{chan.port}/')
test_eq((r.status_code, r.text), (200, 'fastcdp\n'))

In [ ]:
#| export
class ExtCDP(CDP):
    "CDP spoken with a companion extension, as JSON frames over a channel to it"
    @classmethod
    async def connect(cls,
        chan:ExtChannel, # A connected channel to the extension
        debug:bool=None # Print events as they arrive?
    ):
        self = cls(debug=debug)
        self.chan = chan
        self._pump = asyncio.create_task(self._pump_events())
        return self

    @classmethod
    async def listen(cls,
        port:int=34654, # Port the extension dials
        token:str=None, # Shared secret the extension must present
        timeout:float=None, # Max seconds to wait for the extension
        debug:bool=None # Print events as they arrive?
    ):
        "Listen on `port` and speak CDP with the first extension that dials in"
        chan = await ExtChannel.listen(port=port, token=token)
        await chan.wait_peer(timeout)
        return await cls.connect(chan, debug=debug)

    async def _pump_events(self):
        while True:
            msg = obj2dict(await self.chan.events.get())
            if 'method' not in msg: continue
            if (tid := msg.pop('tabId', None)) is not None: msg['sessionId'] = tid
            self._dispatch(msg)

    async def _send(self, msg):
        if (sid := msg.pop('sessionId', None)) is not None: msg['tabId'] = sid
        return obj2dict(await self.chan.request(**msg))

    async def _action(self, action, **kw):
        "Extension-side lifecycle operations that have no CDP equivalent"
        r = await self.chan.request(action=action, **kw)
        if 'error' in r: raise RuntimeError(f"{action}: {r['error']}")
        return obj2dict(r['result'])

    async def close(self):
        self._pump.cancel()
        if (t := getattr(self, '_dialog_task', None)): t.cancel()
        await self.chan.close()

    @property
    def is_open(self): return self.chan.is_open

`connect` wraps any connected channel; `listen` is the common case of waiting for the extension locally. `_send` swaps fastcdp's `sessionId` for the extension's `tabId` on the way out, and `_pump_events` swaps it back on the way in, so every inherited helper that threads `sid` works against tabs directly. `_action` sends the lifecycle frames the extension performs with the `chrome.tabs`/`chrome.debugger` APIs: opening, attaching, listing, and closing tabs.

The `chan` given to `connect` just needs the `ExtChannel` surface (`request`, `events`, `close`, `is_open`), so other transports plug in without subclassing ceremony: dialoghelper's `Channel` is one, which is how solvecdp drives the extension through a solveit relay.

In [ ]:
#| export
class ExtPage(Page):
    "A `Page` whose tab is closed through the extension"
    async def close(self):
        try: await self.cdp._action('close-tab', tabId=self.t)
        except RuntimeError: pass

@patch
async def new_page(self:ExtCDP, url:str='about:blank', active:bool=False):
    "Open a new browser tab and return a `Page` driving it"
    tid = (await self._action('new-tab', url=url, active=active))['tabId']
    await self.page.enable(sid=tid)
    return ExtPage(self, tid, tid)

@patch
async def attach_page(self:ExtCDP, tid:int|str):
    "Attach to an existing tab and return a `Page` driving it; takes the integer `tabId` or hex target `id` from `pages`"
    if isinstance(tid, str):
        tabs = {t['id']:t['tabId'] for t in await self.pages}
        if tid not in tabs: raise KeyError(f'no tab with target id {tid}')
        tid = tabs[tid]
    tid = (await self._action('attach', tabId=tid))['tabId']
    await self.page.enable(sid=tid)
    return ExtPage(self, tid, tid)

@patch(as_prop=True)
async def pages(self:ExtCDP):
    return [t for t in await self._action('get-targets') if t.get('type')=='page']

@patch
async def active_page(self:ExtCDP):
    "A `Page` driving the frontmost tab: the active tab of the last-focused window"
    return await self.attach_page((await self._action('active-tab'))['tabId'])

`new_page`, `attach_page`, and `pages` mirror fastcdp's tab lifecycle through extension actions, returning `Page` proxies so `goto`, `eval`, `ax_tree`, and friends all work unchanged. Continuing with the fake extension from above: a command sent with a `sid` reaches the extension with that `tabId`:

In [ ]:
cdp = await ExtCDP.connect(chan)
r = await cdp('Runtime.evaluate', sid=99, expression='1')
test_eq((r['method'], r['tabId']), ('Runtime.evaluate', 99))

Debugger events cross as `{method, params, tabId}` frames and land in the inherited subscription machinery, with the `tabId` restored to a `sessionId`:

In [ ]:
async with cdp.on('Page.loadEventFired') as q:
    page = await cdp.new_page()
    e = await asyncio.wait_for(q.get(), 5)
test_eq((page.t, e['sessionId']), (99, 99))

`pages` lists the extension's targets, and closing tears down the tab, the pump, and the listener:

In [ ]:
test_eq((await cdp.pages)[0]['type'], 'page')
await page.close()
await cdp.close()
await fake

In [ ]:
#| export
def cdp_yolo():
    "Allow all CDP classes in safepyrun, including the extension transport"
    from pyskills import allow
    fastcdp.core.cdp_yolo()
    allow({ExtChannel:..., ExtCDP:..., ExtPage:...})